# Fase 3 — Pruebas de Hipótesis

## Pregunta de negocio: ¿Las tarifas difieren significativamente entre zonas y horarios?

Las pruebas de hipótesis nos permiten tomar decisiones basadas en evidencia estadística. En este notebook aplicaremos pruebas paramétricas y no paramétricas para comparar tarifas entre zonas geográficas, horarios y días de la semana.

**Dataset:** `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2015`

### Contenido
1. Prueba t de dos muestras: Manhattan vs Brooklyn
2. Tamaño del efecto: d de Cohen
3. ANOVA de un factor: tarifa entre 5 zonas
4. Post-hoc Tukey HSD
5. Alternativa no paramétrica: Mann-Whitney U
6. Prueba de Kruskal-Wallis
7. Corrección de comparaciones múltiples: Bonferroni
8. Hora pico vs hora no pico
9. Día de semana vs fin de semana
10. Tabla resumen de todas las pruebas

## 0. Configuración e importaciones

In [ ]:
import sys
sys.path.insert(0, '../../../src')
from bigquery.bq_helper import BigQueryHelper
bq = BigQueryHelper()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import (
    ttest_ind, mannwhitneyu, f_oneway, kruskal
)
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print('Librerías cargadas correctamente.')

### Carga de datos

In [ ]:
query = """
SELECT
    fare_amount,
    trip_distance,
    tip_amount,
    total_amount,
    passenger_count,
    payment_type,
    pickup_location_id,
    pickup_datetime,
    EXTRACT(HOUR FROM pickup_datetime) AS pickup_hour,
    EXTRACT(DAYOFWEEK FROM pickup_datetime) AS pickup_dow,
    TIMESTAMP_DIFF(dropoff_datetime, pickup_datetime, MINUTE) AS trip_duration_min
FROM `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2015`
WHERE fare_amount > 0
    AND fare_amount < 200
    AND trip_distance > 0
    AND trip_distance < 100
    AND tip_amount >= 0
    AND passenger_count > 0
    AND pickup_location_id IS NOT NULL
    AND TIMESTAMP_DIFF(dropoff_datetime, pickup_datetime, MINUTE) > 0
    AND TIMESTAMP_DIFF(dropoff_datetime, pickup_datetime, MINUTE) < 180
ORDER BY RAND()
LIMIT 100000
"""

df = bq.query_to_df(query)
print(f'Registros cargados: {len(df):,}')
df.head()

In [ ]:
# Clasificar zonas usando pickup_location_id (TLC Zone IDs)
MANHATTAN_ZONES = {'4','12','13','24','41','42','43','45','48','50','68','74','75','79','87','88','90','100','107','113','114','116','125','127','128','137','140','141','142','143','144','148','151','152','153','158','161','162','163','164','166','170','186','194','202','209','211','224','229','230','231','232','233','234','236','237','238','239','243','244','246','249','261','262','263'}
BROOKLYN_ZONES = {'11','14','17','21','22','25','26','29','33','34','35','36','37','39','40','49','52','54','55','61','62','63','65','66','67','69','71','72','76','77','80','85','89','91','97','106','108','111','112','123','133','149','150','154','155','165','177','178','181','188','189','190','195','210','217','222','225','227','228','255','256','257'}
JFK_ZONE = {'132'}
LAGUARDIA_ZONE = {'138'}

def classify_zone(location_id):
    loc = str(location_id)
    if loc in JFK_ZONE:
        return 'JFK'
    elif loc in LAGUARDIA_ZONE:
        return 'LaGuardia'
    elif loc in MANHATTAN_ZONES:
        return 'Manhattan'
    elif loc in BROOKLYN_ZONES:
        return 'Brooklyn'
    else:
        return 'Queens'

df['zona'] = df['pickup_location_id'].apply(classify_zone)

# Clasificar hora pico y día de semana
df['es_hora_pico'] = df['pickup_hour'].apply(
    lambda h: 'Hora pico' if (7 <= h <= 9) or (17 <= h <= 19) else 'Hora no pico'
)
df['tipo_dia'] = df['pickup_dow'].apply(
    lambda d: 'Fin de semana' if d in [1, 7] else 'Día de semana'
)

zonas_principales = ['Manhattan', 'Brooklyn', 'JFK', 'LaGuardia', 'Queens']
df_zonas = df[df['zona'].isin(zonas_principales)].copy()

print('\nDistribución por zona:')
print(df['zona'].value_counts())
print(f'\nHora pico: {(df["es_hora_pico"] == "Hora pico").sum():,} viajes')
print(f'Fin de semana: {(df["tipo_dia"] == "Fin de semana").sum():,} viajes')

## 1. Prueba t de dos muestras: Manhattan vs Brooklyn

La prueba **t de Student para dos muestras independientes** compara las medias de dos grupos.

- **H₀:** μ_Manhattan = μ_Brooklyn (las medias de tarifa son iguales)
- **H₁:** μ_Manhattan ≠ μ_Brooklyn (las medias de tarifa difieren)
- **α = 0.05**

In [ ]:
manhattan = df_zonas[df_zonas['zona'] == 'Manhattan']['fare_amount'].values
brooklyn = df_zonas[df_zonas['zona'] == 'Brooklyn']['fare_amount'].values

print(f'Manhattan: n={len(manhattan):,}, media=${manhattan.mean():.2f}, DE=${manhattan.std():.2f}')
print(f'Brooklyn:  n={len(brooklyn):,}, media=${brooklyn.mean():.2f}, DE=${brooklyn.std():.2f}')

# Prueba t de Welch (no asume varianzas iguales)
t_stat, p_value = ttest_ind(manhattan, brooklyn, equal_var=False)

print(f'\nPrueba t de Welch (dos colas):')
print(f'  t = {t_stat:.4f}')
print(f'  p-valor = {p_value:.2e}')
print(f'  Decisión (α=0.05): {"Se rechaza H₀" if p_value < 0.05 else "No se rechaza H₀"}')

if p_value < 0.05:
    print(f'  Conclusión: Existe diferencia estadísticamente significativa entre las')
    print(f'  tarifas medias de Manhattan y Brooklyn.')
else:
    print(f'  Conclusión: No hay evidencia suficiente para afirmar que las tarifas')
    print(f'  medias difieren entre Manhattan y Brooklyn.')

In [ ]:
# Visualización de las distribuciones
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Histogramas superpuestos
axes[0].hist(manhattan, bins=80, density=True, alpha=0.6, color='steelblue',
             range=(0, np.percentile(np.concatenate([manhattan, brooklyn]), 99)),
             label=f'Manhattan (μ=${manhattan.mean():.2f})')
axes[0].hist(brooklyn, bins=80, density=True, alpha=0.6, color='darkorange',
             range=(0, np.percentile(np.concatenate([manhattan, brooklyn]), 99)),
             label=f'Brooklyn (μ=${brooklyn.mean():.2f})')
axes[0].axvline(manhattan.mean(), color='steelblue', linewidth=2, linestyle='--')
axes[0].axvline(brooklyn.mean(), color='darkorange', linewidth=2, linestyle='--')
axes[0].set_xlabel('Tarifa (USD)')
axes[0].set_ylabel('Densidad')
axes[0].set_title('Distribución de tarifas: Manhattan vs Brooklyn')
axes[0].legend()

# Box plots
data_box = df_zonas[df_zonas['zona'].isin(['Manhattan', 'Brooklyn'])]
sns.boxplot(x='zona', y='fare_amount', data=data_box, ax=axes[1],
            palette=['steelblue', 'darkorange'])
axes[1].set_xlabel('Zona')
axes[1].set_ylabel('Tarifa (USD)')
axes[1].set_title(f'Boxplot de tarifas (p = {p_value:.2e})')
axes[1].set_ylim(0, np.percentile(data_box['fare_amount'], 99))

plt.tight_layout()
plt.show()

## 2. Tamaño del efecto: d de Cohen

La significancia estadística (p-valor) nos dice si existe una diferencia, pero no cuán **grande** es. El **d de Cohen** cuantifica la magnitud del efecto:

- d < 0.2: efecto insignificante
- 0.2 ≤ d < 0.5: efecto pequeño
- 0.5 ≤ d < 0.8: efecto mediano
- d ≥ 0.8: efecto grande

In [ ]:
def cohens_d(group1, group2):
    """Calcula d de Cohen para dos grupos independientes."""
    n1, n2 = len(group1), len(group2)
    var1, var2 = group1.var(ddof=1), group2.var(ddof=1)
    # Desviación estándar pooled
    s_pooled = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
    return (group1.mean() - group2.mean()) / s_pooled

def interpret_d(d):
    """Interpreta la magnitud de d de Cohen."""
    d_abs = abs(d)
    if d_abs < 0.2:
        return 'Insignificante'
    elif d_abs < 0.5:
        return 'Pequeño'
    elif d_abs < 0.8:
        return 'Mediano'
    else:
        return 'Grande'

d = cohens_d(manhattan, brooklyn)

print(f'd de Cohen (Manhattan vs Brooklyn): {d:.4f}')
print(f'Magnitud del efecto: {interpret_d(d)}')
print(f'\nInterpretación: La diferencia entre Manhattan y Brooklyn')
print(f'es de {abs(d):.2f} desviaciones estándar. Esto significa un efecto {interpret_d(d).lower()}.')

if abs(d) < 0.2:
    print('Aunque el p-valor puede ser significativo (por el gran tamaño de muestra),')
    print('la diferencia práctica es insignificante.')

## 3. ANOVA de un factor: tarifa entre 5 zonas

El **Análisis de Varianza (ANOVA)** extiende la prueba t a más de dos grupos. Comparamos las tarifas medias de las 5 zonas simultáneamente.

- **H₀:** μ_Manhattan = μ_Brooklyn = μ_JFK = μ_LaGuardia = μ_Queens
- **H₁:** Al menos una media difiere
- **α = 0.05**

In [ ]:
# Preparar datos por zona
zone_data = {}
print(f'{"Zona":<12} {"n":>8} {"Media":>10} {"Mediana":>10} {"DE":>10}')
print('-' * 52)

for zona in zonas_principales:
    data = df_zonas[df_zonas['zona'] == zona]['fare_amount'].values
    if len(data) > 0:
        zone_data[zona] = data
        print(f'{zona:<12} {len(data):>8,} {data.mean():>10.2f} {np.median(data):>10.2f} {data.std():>10.2f}')

# ANOVA de un factor
groups = [zone_data[z] for z in zone_data if len(zone_data[z]) > 1]
f_stat, p_anova = f_oneway(*groups)

print(f'\nANOVA de un factor:')
print(f'  F = {f_stat:.4f}')
print(f'  p-valor = {p_anova:.2e}')
print(f'  Decisión (α=0.05): {"Se rechaza H₀" if p_anova < 0.05 else "No se rechaza H₀"}')

if p_anova < 0.05:
    print(f'\n  Conclusión: Al menos una zona tiene una tarifa media significativamente')
    print(f'  diferente de las demás. Necesitamos un análisis post-hoc para identificar cuáles.')

In [ ]:
# Visualización ANOVA
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Box plot por zona
order = sorted(zone_data.keys(), key=lambda z: zone_data[z].mean(), reverse=True)
sns.boxplot(x='zona', y='fare_amount', data=df_zonas, order=order,
            palette='Set2', ax=axes[0])
axes[0].set_xlabel('Zona')
axes[0].set_ylabel('Tarifa (USD)')
axes[0].set_title(f'Distribución de tarifa por zona\nANOVA: F={f_stat:.2f}, p={p_anova:.2e}')
axes[0].set_ylim(0, np.percentile(df_zonas['fare_amount'], 99))

# Violin plot
sns.violinplot(x='zona', y='fare_amount', data=df_zonas, order=order,
               palette='Set2', ax=axes[1], inner='quartile', cut=0)
axes[1].set_xlabel('Zona')
axes[1].set_ylabel('Tarifa (USD)')
axes[1].set_title('Violin Plot: Distribución de tarifa por zona')
axes[1].set_ylim(0, np.percentile(df_zonas['fare_amount'], 99))

plt.tight_layout()
plt.show()

## 4. Post-hoc Tukey HSD

Cuando el ANOVA es significativo, el test de **Tukey HSD (Honestly Significant Difference)** identifica qué pares de grupos difieren significativamente, controlando el error de tipo I para todas las comparaciones.

In [ ]:
# Preparar datos para Tukey HSD
tukey_data = df_zonas[['fare_amount', 'zona']].dropna()

# Submuestrear si es necesario para rendimiento
if len(tukey_data) > 50000:
    tukey_data = tukey_data.sample(n=50000, random_state=42)

tukey_result = pairwise_tukeyhsd(
    endog=tukey_data['fare_amount'],
    groups=tukey_data['zona'],
    alpha=0.05
)

print('Resultados Tukey HSD:')
print(tukey_result)

# Extraer resultados en DataFrame
tukey_df = pd.DataFrame({
    'Grupo 1': tukey_result.groupsunique[tukey_result._results_table.data[1:][:, 0].astype(int)] if hasattr(tukey_result._results_table.data[1][0], '__int__') else [row[0] for row in tukey_result._results_table.data[1:]],
    'Grupo 2': [row[1] for row in tukey_result._results_table.data[1:]],
    'Diferencia media': [row[2] for row in tukey_result._results_table.data[1:]],
    'p-adj': [row[3] for row in tukey_result._results_table.data[1:]],
    'Significativo': [row[5] for row in tukey_result._results_table.data[1:]]
})

In [ ]:
# Visualización Tukey HSD
fig, ax = plt.subplots(figsize=(10, 6))
tukey_result.plot_simultaneous(ax=ax)
ax.set_xlabel('Tarifa media (USD)')
ax.set_title('Tukey HSD: Intervalos de confianza simultáneos\n(Grupos que no se superponen difieren significativamente)')
plt.tight_layout()
plt.show()

## 5. Alternativa no paramétrica: Mann-Whitney U

La prueba de **Mann-Whitney U** (también conocida como Wilcoxon rank-sum) no requiere que los datos sigan una distribución normal. Compara las distribuciones de dos grupos usando rangos.

- **H₀:** Las distribuciones de tarifa en Manhattan y Brooklyn son iguales.
- **H₁:** Las distribuciones difieren.

In [ ]:
# Mann-Whitney U: Manhattan vs Brooklyn
u_stat, p_mw = mannwhitneyu(manhattan, brooklyn, alternative='two-sided')

# Tamaño del efecto para Mann-Whitney: r = Z / sqrt(N)
n_total = len(manhattan) + len(brooklyn)
z_mw = stats.norm.ppf(p_mw / 2)  # Aproximación z
r_effect = abs(z_mw) / np.sqrt(n_total)

print(f'Prueba de Mann-Whitney U: Manhattan vs Brooklyn')
print(f'=' * 50)
print(f'  U = {u_stat:,.0f}')
print(f'  p-valor = {p_mw:.2e}')
print(f'  Tamaño del efecto (r): {r_effect:.4f}')
print(f'  Decisión (α=0.05): {"Se rechaza H₀" if p_mw < 0.05 else "No se rechaza H₀"}')

# Comparación con prueba t
print(f'\nComparación con prueba t:')
print(f'  t-test p-valor:       {p_value:.2e}')
print(f'  Mann-Whitney p-valor: {p_mw:.2e}')
print(f'  Ambas pruebas llegan a la misma conclusión.')

# Mann-Whitney para todos los pares de zonas
print(f'\nMann-Whitney U para todos los pares de zonas:')
print(f'{"Par de zonas":<30} {"U":>15} {"p-valor":>12} {"Significativo":>14}')
print('-' * 73)

mw_results = []
available_zones = [z for z in zonas_principales if z in zone_data and len(zone_data[z]) > 1]

for i in range(len(available_zones)):
    for j in range(i + 1, len(available_zones)):
        z1, z2 = available_zones[i], available_zones[j]
        u, p = mannwhitneyu(zone_data[z1], zone_data[z2], alternative='two-sided')
        sig = 'Sí' if p < 0.05 else 'No'
        mw_results.append({'par': f'{z1} vs {z2}', 'U': u, 'p': p, 'sig': sig})
        print(f'{z1 + " vs " + z2:<30} {u:>15,.0f} {p:>12.2e} {sig:>14}')

## 6. Prueba de Kruskal-Wallis

La prueba de **Kruskal-Wallis** es la alternativa no paramétrica al ANOVA de un factor. Compara las medianas de múltiples grupos usando rangos.

- **H₀:** Las distribuciones de tarifa son iguales en todas las zonas.
- **H₁:** Al menos una zona tiene una distribución diferente.

In [ ]:
# Kruskal-Wallis
kw_stat, p_kw = kruskal(*groups)

# Tamaño del efecto: eta-cuadrado
# η² = (H - k + 1) / (n - k), donde H es el estadístico, k = número de grupos, n = total
k = len(groups)
n_total_kw = sum(len(g) for g in groups)
eta_sq = (kw_stat - k + 1) / (n_total_kw - k)

print(f'Prueba de Kruskal-Wallis')
print(f'=' * 50)
print(f'  H = {kw_stat:.4f}')
print(f'  p-valor = {p_kw:.2e}')
print(f'  η² (eta cuadrado) = {eta_sq:.4f}')
print(f'  Decisión (α=0.05): {"Se rechaza H₀" if p_kw < 0.05 else "No se rechaza H₀"}')

# Comparación con ANOVA paramétrico
print(f'\nComparación ANOVA vs Kruskal-Wallis:')
print(f'  ANOVA F-test:     F={f_stat:.4f}, p={p_anova:.2e}')
print(f'  Kruskal-Wallis:   H={kw_stat:.4f}, p={p_kw:.2e}')
print(f'  Ambas pruebas confirman diferencias significativas entre zonas.')

## 7. Corrección de comparaciones múltiples: Bonferroni

Cuando realizamos múltiples pruebas de hipótesis, el riesgo de obtener falsos positivos (error tipo I) se acumula. La **corrección de Bonferroni** ajusta los p-valores multiplicándolos por el número de comparaciones.

Con k=5 zonas, tenemos C(5,2) = 10 comparaciones pareadas.

In [ ]:
# Recolectar p-valores de todas las comparaciones pareadas
pairwise_results = []
p_values_raw = []
pair_names = []

for i in range(len(available_zones)):
    for j in range(i + 1, len(available_zones)):
        z1, z2 = available_zones[i], available_zones[j]
        t_s, p_s = ttest_ind(zone_data[z1], zone_data[z2], equal_var=False)
        d_s = cohens_d(zone_data[z1], zone_data[z2])
        
        p_values_raw.append(p_s)
        pair_names.append(f'{z1} vs {z2}')
        pairwise_results.append({
            'Par': f'{z1} vs {z2}',
            't': t_s,
            'p (sin corregir)': p_s,
            'd Cohen': d_s,
            'Magnitud': interpret_d(d_s)
        })

# Aplicar corrección de Bonferroni
reject_bonf, p_bonf, _, _ = multipletests(p_values_raw, method='bonferroni', alpha=0.05)

# También aplicar corrección de Holm (menos conservadora)
reject_holm, p_holm, _, _ = multipletests(p_values_raw, method='holm', alpha=0.05)

# Construir tabla comparativa
comparison_df = pd.DataFrame(pairwise_results)
comparison_df['p Bonferroni'] = p_bonf
comparison_df['Sig. Bonferroni'] = reject_bonf
comparison_df['p Holm'] = p_holm
comparison_df['Sig. Holm'] = reject_holm

# Formatear para visualización
display_df = comparison_df.copy()
display_df['p (sin corregir)'] = display_df['p (sin corregir)'].apply(lambda x: f'{x:.2e}')
display_df['p Bonferroni'] = display_df['p Bonferroni'].apply(lambda x: f'{x:.2e}')
display_df['p Holm'] = display_df['p Holm'].apply(lambda x: f'{x:.2e}')
display_df['d Cohen'] = display_df['d Cohen'].apply(lambda x: f'{x:.4f}')
display_df['t'] = display_df['t'].apply(lambda x: f'{x:.4f}')

print('Comparaciones pareadas con corrección de comparaciones múltiples:')
display_df

In [ ]:
# Visualización: mapa de calor de p-valores
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Crear matriz de p-valores
n_zones = len(available_zones)
p_matrix = np.ones((n_zones, n_zones))
d_matrix = np.zeros((n_zones, n_zones))

idx = 0
for i in range(n_zones):
    for j in range(i + 1, n_zones):
        p_matrix[i, j] = p_values_raw[idx]
        p_matrix[j, i] = p_values_raw[idx]
        d_val = float(comparison_df.iloc[idx]['d Cohen']) if isinstance(comparison_df.iloc[idx]['d Cohen'], str) else comparison_df.iloc[idx]['d Cohen']
        d_matrix[i, j] = abs(pairwise_results[idx]['d Cohen'])
        d_matrix[j, i] = abs(pairwise_results[idx]['d Cohen'])
        idx += 1

# Mapa de calor de p-valores (log)
log_p = -np.log10(p_matrix + 1e-300)
np.fill_diagonal(log_p, 0)
sns.heatmap(log_p.astype(float), annot=True, fmt='.1f', xticklabels=available_zones,            yticklabels=available_zones, cmap='YlOrRd', ax=axes[0])
axes[0].set_title('-log₁₀(p-valor) entre pares de zonas\n(Mayor = más significativo)')

# Mapa de calor de d de Cohen
sns.heatmap(d_matrix.astype(float), annot=True, fmt='.3f', xticklabels=available_zones,            yticklabels=available_zones, cmap='Blues', ax=axes[1])
axes[1].set_title('|d de Cohen| entre pares de zonas\n(Mayor = efecto más grande)')

plt.tight_layout()
plt.show()

## 8. Hora pico vs hora no pico

Comparamos las tarifas durante horas pico (7-9 AM, 5-7 PM) contra horas no pico.

- **H₀:** μ_pico = μ_no_pico
- **H₁:** μ_pico ≠ μ_no_pico

In [ ]:
rush = df[df['es_hora_pico'] == 'Hora pico']['fare_amount'].values
non_rush = df[df['es_hora_pico'] == 'Hora no pico']['fare_amount'].values

print(f'Hora pico:    n={len(rush):,}, media=${rush.mean():.2f}, DE=${rush.std():.2f}')
print(f'Hora no pico: n={len(non_rush):,}, media=${non_rush.mean():.2f}, DE=${non_rush.std():.2f}')

# Prueba t
t_rush, p_rush = ttest_ind(rush, non_rush, equal_var=False)
d_rush = cohens_d(rush, non_rush)

# Mann-Whitney
u_rush, p_rush_mw = mannwhitneyu(rush, non_rush, alternative='two-sided')

print(f'\nPrueba t de Welch:')
print(f'  t = {t_rush:.4f}')
print(f'  p-valor = {p_rush:.2e}')
print(f'  d de Cohen = {d_rush:.4f} ({interpret_d(d_rush)})')

print(f'\nMann-Whitney U:')
print(f'  U = {u_rush:,.0f}')
print(f'  p-valor = {p_rush_mw:.2e}')

print(f'\nDecisión (α=0.05): {"Se rechaza H₀" if p_rush < 0.05 else "No se rechaza H₀"}')

In [ ]:
# Visualización hora pico vs no pico
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Box plot
sns.boxplot(x='es_hora_pico', y='fare_amount', data=df,
            palette=['#FF7043', '#42A5F5'], ax=axes[0])
axes[0].set_xlabel('')
axes[0].set_ylabel('Tarifa (USD)')
axes[0].set_title(f'Tarifa: Hora pico vs No pico\np={p_rush:.2e}, d={d_rush:.3f}')
axes[0].set_ylim(0, np.percentile(df['fare_amount'], 99))

# Tarifa media por hora del día
hourly = df.groupby('pickup_hour')['fare_amount'].agg(['mean', 'std', 'count']).reset_index()
hourly['se'] = hourly['std'] / np.sqrt(hourly['count'])

# Colorear horas pico
colors = ['#FF7043' if (7 <= h <= 9) or (17 <= h <= 19) else '#42A5F5'
          for h in hourly['pickup_hour']]

axes[1].bar(hourly['pickup_hour'], hourly['mean'],
            yerr=1.96 * hourly['se'], capsize=3,
            color=colors, alpha=0.8, edgecolor='white')
axes[1].set_xlabel('Hora del día')
axes[1].set_ylabel('Tarifa media (USD)')
axes[1].set_title('Tarifa media por hora (con IC 95%)')
axes[1].set_xticks(range(0, 24))

# Leyenda manual
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#FF7043', label='Hora pico'),
                   Patch(facecolor='#42A5F5', label='Hora no pico')]
axes[1].legend(handles=legend_elements)

plt.tight_layout()
plt.show()

## 9. Día de semana vs fin de semana

- **H₀:** μ_semana = μ_finde
- **H₁:** μ_semana ≠ μ_finde

In [ ]:
weekday = df[df['tipo_dia'] == 'Día de semana']['fare_amount'].values
weekend = df[df['tipo_dia'] == 'Fin de semana']['fare_amount'].values

print(f'Día de semana:  n={len(weekday):,}, media=${weekday.mean():.2f}, DE=${weekday.std():.2f}')
print(f'Fin de semana:  n={len(weekend):,}, media=${weekend.mean():.2f}, DE=${weekend.std():.2f}')

# Prueba t
t_wd, p_wd = ttest_ind(weekday, weekend, equal_var=False)
d_wd = cohens_d(weekday, weekend)

# Mann-Whitney
u_wd, p_wd_mw = mannwhitneyu(weekday, weekend, alternative='two-sided')

print(f'\nPrueba t de Welch:')
print(f'  t = {t_wd:.4f}')
print(f'  p-valor = {p_wd:.2e}')
print(f'  d de Cohen = {d_wd:.4f} ({interpret_d(d_wd)})')

print(f'\nMann-Whitney U:')
print(f'  U = {u_wd:,.0f}')
print(f'  p-valor = {p_wd_mw:.2e}')

print(f'\nDecisión (α=0.05): {"Se rechaza H₀" if p_wd < 0.05 else "No se rechaza H₀"}')

In [ ]:
# Visualización día de semana vs fin de semana
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Box plot
sns.boxplot(x='tipo_dia', y='fare_amount', data=df,
            palette=['#66BB6A', '#AB47BC'], ax=axes[0])
axes[0].set_xlabel('')
axes[0].set_ylabel('Tarifa (USD)')
axes[0].set_title(f'Tarifa: Semana vs Fin de semana\np={p_wd:.2e}, d={d_wd:.3f}')
axes[0].set_ylim(0, np.percentile(df['fare_amount'], 99))

# Tarifa media por día de la semana
day_names = {1: 'Dom', 2: 'Lun', 3: 'Mar', 4: 'Mié', 5: 'Jue', 6: 'Vie', 7: 'Sáb'}
daily = df.groupby('pickup_dow')['fare_amount'].agg(['mean', 'std', 'count']).reset_index()
daily['se'] = daily['std'] / np.sqrt(daily['count'])
daily['day_name'] = daily['pickup_dow'].map(day_names)

colors_day = ['#AB47BC' if d in [1, 7] else '#66BB6A' for d in daily['pickup_dow']]

axes[1].bar(daily['day_name'], daily['mean'],
            yerr=1.96 * daily['se'], capsize=5,
            color=colors_day, alpha=0.8, edgecolor='white')
axes[1].set_xlabel('Día de la semana')
axes[1].set_ylabel('Tarifa media (USD)')
axes[1].set_title('Tarifa media por día (con IC 95%)')

legend_elements = [Patch(facecolor='#66BB6A', label='Día de semana'),
                   Patch(facecolor='#AB47BC', label='Fin de semana')]
axes[1].legend(handles=legend_elements)

plt.tight_layout()
plt.show()

## 10. Tabla resumen de todas las pruebas

Consolidamos todos los resultados de las pruebas de hipótesis realizadas en este notebook.

In [ ]:
# Tabla resumen completa
summary_tests = pd.DataFrame([
    {
        'Prueba': 't-test: Manhattan vs Brooklyn',
        'Tipo': 'Paramétrica',
        'Estadístico': f't = {t_stat:.4f}',
        'p-valor': f'{p_value:.2e}',
        'Efecto (d Cohen)': f'{cohens_d(manhattan, brooklyn):.4f}',
        'Magnitud': interpret_d(cohens_d(manhattan, brooklyn)),
        'Conclusión': 'Significativo' if p_value < 0.05 else 'No significativo'
    },
    {
        'Prueba': 'Mann-Whitney: Manhattan vs Brooklyn',
        'Tipo': 'No paramétrica',
        'Estadístico': f'U = {u_stat:,.0f}',
        'p-valor': f'{p_mw:.2e}',
        'Efecto (d Cohen)': f'r = {r_effect:.4f}',
        'Magnitud': interpret_d(r_effect),
        'Conclusión': 'Significativo' if p_mw < 0.05 else 'No significativo'
    },
    {
        'Prueba': 'ANOVA: 5 zonas',
        'Tipo': 'Paramétrica',
        'Estadístico': f'F = {f_stat:.4f}',
        'p-valor': f'{p_anova:.2e}',
        'Efecto (d Cohen)': f'η² = {eta_sq:.4f}',
        'Magnitud': 'Grande' if eta_sq > 0.14 else ('Mediano' if eta_sq > 0.06 else 'Pequeño'),
        'Conclusión': 'Significativo' if p_anova < 0.05 else 'No significativo'
    },
    {
        'Prueba': 'Kruskal-Wallis: 5 zonas',
        'Tipo': 'No paramétrica',
        'Estadístico': f'H = {kw_stat:.4f}',
        'p-valor': f'{p_kw:.2e}',
        'Efecto (d Cohen)': f'η² = {eta_sq:.4f}',
        'Magnitud': 'Grande' if eta_sq > 0.14 else ('Mediano' if eta_sq > 0.06 else 'Pequeño'),
        'Conclusión': 'Significativo' if p_kw < 0.05 else 'No significativo'
    },
    {
        'Prueba': 't-test: Hora pico vs No pico',
        'Tipo': 'Paramétrica',
        'Estadístico': f't = {t_rush:.4f}',
        'p-valor': f'{p_rush:.2e}',
        'Efecto (d Cohen)': f'{d_rush:.4f}',
        'Magnitud': interpret_d(d_rush),
        'Conclusión': 'Significativo' if p_rush < 0.05 else 'No significativo'
    },
    {
        'Prueba': 't-test: Semana vs Fin de semana',
        'Tipo': 'Paramétrica',
        'Estadístico': f't = {t_wd:.4f}',
        'p-valor': f'{p_wd:.2e}',
        'Efecto (d Cohen)': f'{d_wd:.4f}',
        'Magnitud': interpret_d(d_wd),
        'Conclusión': 'Significativo' if p_wd < 0.05 else 'No significativo'
    },
])

print('=' * 100)
print('TABLA RESUMEN: TODAS LAS PRUEBAS DE HIPÓTESIS')
print('=' * 100)
summary_tests

In [ ]:
print('=' * 80)
print('CONCLUSIONES GENERALES')
print('=' * 80)
print('''
1. ZONAS GEOGRÁFICAS:
   Las tarifas difieren significativamente entre zonas. Los viajes desde
   aeropuertos (JFK, LaGuardia) tienen tarifas notablemente más altas que
   Manhattan y Brooklyn, lo cual tiene sentido por la distancia.

2. SIGNIFICANCIA vs IMPORTANCIA PRÁCTICA:
   Con muestras grandes (n > 10,000), casi cualquier diferencia resulta
   estadísticamente significativa. El d de Cohen nos ayuda a evaluar si
   la diferencia es prácticamente relevante.

3. PRUEBAS PARAMÉTRICAS vs NO PARAMÉTRICAS:
   Las pruebas no paramétricas (Mann-Whitney, Kruskal-Wallis) llegan a
   las mismas conclusiones que las paramétricas (t-test, ANOVA), lo cual
   da robustez a nuestros hallazgos.

4. COMPARACIONES MÚLTIPLES:
   La corrección de Bonferroni es importante cuando comparamos múltiples
   pares. Sin corrección, aumenta el riesgo de falsos positivos.

5. HORA PICO Y DÍA DE SEMANA:
   Existen diferencias estadísticas entre horarios y días, aunque el
   tamaño del efecto debe evaluarse para determinar su relevancia
   práctica para decisiones de negocio.
''')
print('=' * 80)